# SLAYER-OCR layout teacher: qwen3-vl-4b
Use a fresh GPU runtime for exactly one teacher. This notebook is locked to `PAGES = 60`.
The run uses pinned public train pages and pinned model revisions. Reference text is never sent to a teacher. The downloaded private evidence ZIP contains proposals, provenance and checksums, but no page images or reference text. Nothing is published automatically.


In [ ]:
TEACHER_ID = 'qwen3-vl-4b'
PAGES = 60

assert TEACHER_ID == 'qwen3-vl-4b'
assert PAGES == 60
print(f"Locked full run: teacher={TEACHER_ID} pages={PAGES}")


In [ ]:
import subprocess
import sys

PACKAGES = {
    "qwen3-vl-4b": [
        "huggingface_hub==0.36.2", "pillow==11.3.0",
        "transformers==4.57.6", "tokenizers==0.22.2", "bitsandbytes==0.48.1",
        "accelerate==1.13.0", "sentencepiece==0.2.1",
    ],
    "doclayout-yolo": [
        "huggingface_hub==0.36.2", "pillow==11.3.0", "doclayout-yolo==0.0.4",
    ],
    "surya-layout2": [
        "surya-ocr==0.22.1", "transformers==5.12.1",
        "huggingface_hub==1.5.0", "pillow==10.4.0",
    ],
}
RESET = {
    "qwen3-vl-4b": ["torchao", "transformers", "tokenizers", "huggingface_hub"],
    "doclayout-yolo": [],
    "surya-layout2": ["transformers", "tokenizers", "huggingface_hub", "pillow"],
}
if RESET[TEACHER_ID]:
    subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "-y", *RESET[TEACHER_ID]],
        check=False,
    )
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-cache-dir",
     *PACKAGES[TEACHER_ID]],
    check=True,
)
print("INSTALL_OK", TEACHER_ID)


In [ ]:
import importlib.metadata
import json
import os
from pathlib import Path
import subprocess
import sys

CODE_REVISION = 'd42ae1eed9a076e3508329482978eb3dfe61e1b8'
EXPECTED_CONFIG = {'schema': 'slayer-layout-teacher-pilot-config-v1', 'status': 'frozen before teacher inference', 'frozen_at': '2026-09-28', 'scope': 'private weak-label pilot on historical Polish train pages; not benchmark evaluation', 'dataset': {'repo': 'PiotrSty/impact-psnc-polish-ocr', 'revision': 'c7cb156fb95d2880699c33725bbaf1fbc1008fea', 'metadata_path': 'pages/train/metadata.jsonl', 'metadata_sha256': '991ef94a2e2fb3ac519ec3582fa588b1a75a92a87b1b002468cf7c31d3c10427', 'available_pages': 65, 'selected_pages': 60, 'collections': 22, 'license': 'CC-BY-3.0', 'selection': 'collection-round-robin after SHA-256 rank', 'rank_salt': 'slayer-layout-teacher-pilot-v1'}, 'ontology': ['text_region', 'heading', 'table', 'figure', 'caption', 'marginalia', 'header', 'footer', 'page_number'], 'teachers': {'qwen3-vl-4b': {'model_repo': 'Qwen/Qwen3-VL-4B-Instruct', 'revision': 'ebb281ec70b05090aa6165b016eac8ec08e71b17', 'license': 'Apache-2.0', 'package': 'transformers==4.57.6', 'score_kind': 'neutral-unavailable', 'score': 0.5, 'max_new_tokens': 4096}, 'doclayout-yolo': {'model_repo': 'juliozhao/DocLayout-YOLO-DocStructBench', 'revision': '8c3299a30b8ff29a1503c4431b035b93220f7b11', 'license': 'Apache-2.0 weights; AGPL-3.0 inference package', 'package': 'doclayout-yolo==0.0.4', 'score_kind': 'model-confidence', 'threshold': 0.25, 'nms_iou': 0.45, 'image_size': 1024}, 'surya-layout2': {'model_repo': 'datalab-to/surya_layout2', 'revision': '0aee81d5fd9275c0582e545bf3a56944b1e75679', 'license': 'AI Pubs OpenRAIL-M', 'package': 'surya-ocr==0.22.1', 'score_kind': 'model-confidence when exposed; otherwise neutral-unavailable'}}, 'consensus': {'categories': ['text_region', 'heading', 'table', 'figure', 'caption', 'marginalia', 'header', 'footer', 'page_number'], 'quorum': 2, 'iou_threshold': 0.5, 'conflict_iou': 0.5, 'min_score': 0.0}, 'boundaries': {'test_or_validation_pages_used': 0, 'reference_text_sent_to_teachers': False, 'historical_spelling_modernized': False, 'teacher_outputs_are_ground_truth': False, 'images_or_references_in_evidence_zip': False, 'automatic_publication': False, 'release_requires_license_review': True}, 'claim_boundary': 'This pilot measures teacher agreement and review load. It does not establish label accuracy, RF-DETR improvement, or OCR SOTA.'}

repo = Path("/content/OCR_engine")
if not repo.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/PiotrStyla/OCR_engine.git", str(repo)],
        check=True,
    )
subprocess.run(["git", "-C", str(repo), "fetch", "origin", CODE_REVISION], check=True)
subprocess.run(
    ["git", "-C", str(repo), "checkout", "--detach", CODE_REVISION], check=True
)
resolved = subprocess.check_output(
    ["git", "-C", str(repo), "rev-parse", "HEAD"], text=True
).strip()
assert resolved == CODE_REVISION

config_path = repo / "experiments/2026-09-28/slayer-layout-teacher-pilot-v1/config.json"
CONFIG = json.loads(config_path.read_text(encoding="utf-8"))
assert CONFIG == EXPECTED_CONFIG
os.chdir(repo)
sys.path.insert(0, str(repo))

import torch
assert torch.cuda.is_available(), "Select a GPU runtime, then Run all."
expected_package = CONFIG["teachers"][TEACHER_ID]["package"].split("==")
assert importlib.metadata.version(expected_package[0]) == expected_package[1]
EXPECTED_RUNTIME = {
    "qwen3-vl-4b": {"huggingface_hub": "0.36.2", "Pillow": "11.3.0"},
    "doclayout-yolo": {"huggingface_hub": "0.36.2", "Pillow": "11.3.0"},
    "surya-layout2": {
        "huggingface_hub": "1.5.0", "Pillow": "10.4.0", "transformers": "5.12.1",
    },
}
for package, version in EXPECTED_RUNTIME[TEACHER_ID].items():
    assert importlib.metadata.version(package) == version, (package, version)
if TEACHER_ID == "qwen3-vl-4b":
    from transformers import AutoModelForImageTextToText
    assert AutoModelForImageTextToText is not None
print("PINNED_RUNTIME_OK", torch.cuda.get_device_name(0), CODE_REVISION)


In [ ]:
from training.slayer_layout_teacher_pilot import run

result_archive = run(TEACHER_ID, CONFIG, pages=PAGES, output_root="/content", code_revision=CODE_REVISION)
print("PRIVATE_EVIDENCE_READY", result_archive)


In [ ]:
from google.colab import files
files.download(str(result_archive))
